In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# import os
# for dir, _, files in os.walk('kaggle/input'):
    

In [3]:
"""
Resources include: 
::dataset: data/sp_data/*.json
::config: config/*
::tokenizer: tokenizer/*json
"""

'\nResources include: \n::dataset: data/sp_data/*.json\n::config: config/*\n::tokenizer: tokenizer/*json\n'

In [4]:
!pip install lightning
# !pip install scikit-learn
# !pip install transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 39.0 MB/s eta 0:00:0000:01


In [5]:
# export all params in params.py here
import os

TRAIN_PATH = 'data/sp_data/train_set.fasta'
BENCHMARK_PATH = 'data/sp_data/benchmark_set_sp5.fasta'
ROOT_DIR = '/kaggle/input/sp-resource7'
BERT_CKPT = '/kaggle/input/bert-checkpoint1'
# ROOT_DIR = os.path.dirname(os.path.abspath(__file__))
# TRANSFORMER_CONFIG_DEFAULT = 'configs/transformer_config_default.json'

EPOCHS = 100

"""
MODEL CONFIGURATION
"""
SP_LABELS = dict(NO_SP=0, SP=1, LIPO=2, TAT=3, PILIN=4, TATLIPO=5)
KINGDOM = dict(EUKARYA=0, POSITIVE=1, NEGATIVE=2, ARCHAEA=1)
BATCH_SIZE = 8
MODEL = 'lstm'
DATA = 'aa'

"""
LOGGER CONFIGURATION
"""
KAGGLE_DIR = '/kaggle/working'
DEFAULT_LOG_DIR = 'logs'


In [6]:
# export Model Checkpoint Callback in callback_utils.py here
from pathlib import Path
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

model_checkpoint = ModelCheckpoint(
    dirpath=str(Path(KAGGLE_DIR)),
    filename=f"{MODEL}_epoch={EPOCHS}",
    enable_version_counter=True,
    monitor='val_loss',
    every_n_epochs=1,
    save_on_train_epoch_end=True,
    save_top_k=1
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    min_delta=0.00,
    patience=3,
    verbose=False,
    mode="max"
)

print("Finished")

Finished


In [7]:
# code SPDataset (extends torch.utils.data.Dataset)
import json
from pathlib import Path
from typing import Literal

import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import GPT2TokenizerFast, BertTokenizer


class SPDataset(Dataset):
    def __init__(self, json_paths: list[str], data_type: Literal['aa', 'smiles'] = 'smiles'):
        df = pd.concat(pd.read_json(path) for path in json_paths)
        self.smiles = df['smiles'].tolist()
        self.aa_seq = df['aa_seq'].tolist()
        self.labels = df['label'].tolist()
        self.kingdoms = df['kingdom'].tolist()
        self.data_type = data_type
        self.config = self.__load_data_config(data_type=data_type)
        tokenizer_path = str(Path(ROOT_DIR) / self.config['tokenizer_path'])
        tokenizer = GPT2TokenizerFast(tokenizer_file=tokenizer_path)
        if tokenizer.pad_token is None:
            tokenizer.add_special_tokens({'pad_token': '[PAD]'})
        self.tokenizer = tokenizer
        if MODEL == 'bert_pretrained':
            self.tokenizer = BertTokenizer.from_pretrained("Rostlab/prot_bert")

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, index):
        kingdom = self.kingdoms[index]
        seq = self.smiles[index] if self.data_type == 'smiles' else self.aa_seq[index]
        label = torch.zeros(len(SP_LABELS), dtype=torch.int64)
        label[SP_LABELS[self.labels[index]]] = 1
        input_ids = self.tokenizer.encode(seq, max_length=self.config['max_len'], padding="max_length", truncation=True)
        return torch.tensor(input_ids, dtype=torch.int64), torch.tensor(label, dtype=torch.int64), kingdom

    @staticmethod
    def __load_data_config(data_type: str):
        with open(str(Path(ROOT_DIR) / f'configs/data_config/{data_type}_config.json')) as f:
            config = json.load(f)
            return config


print("Finished")

Finished


In [14]:
# code model (extend torch.nn.Module)
"""NN layers"""
import math

import torch
from torch import nn


class InputEmbedding(nn.Module):
    def __init__(self, vocab_size: int = 100, d_model: int = 512):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.input_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=d_model
        )

    def forward(self, x):
        x = self.input_embedding(x) * math.sqrt(self.d_model)
        # print(x.shape)
        return x


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int = 512, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        pe_x = self.pe[:, :x.size(1), :].requires_grad_(True)
        x = x + pe_x
        return self.dropout(x)


class TransformerEncoder(nn.Module):
    def __init__(self, d_model: int = 512, nhead: int = 8, num_layers: int = 6):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=num_layers
        )

    def forward(self, x):
        x = self.encoder(x)
        # use average [CLS] token with all other word tokens
        # x = torch.mean(x, dim=1)

        # use only [CLS] token
        x = x[:, 0, :]
        return x


class ConvolutionalEncoder(nn.Module):
    def __init__(
            self,
            embedding_dim: int = 512,
            dropout: float = 0.1,
            kernel_size: int = 3,
            stride: int = 1,
            padding: int = 0,
            n_base: int = 1024,
    ):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        self.conv1 = nn.Sequential(
            nn.Conv1d(in_channels=embedding_dim, out_channels=n_base, kernel_size=kernel_size, stride=stride,
                      padding=padding),
            nn.ReLU(),
            nn.AvgPool1d(kernel_size=2, stride=2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv1d(in_channels=n_base, out_channels=n_base * 4, kernel_size=kernel_size, stride=stride,
                      padding=padding),
            nn.ReLU(),
            nn.AvgPool1d(kernel_size=2, stride=2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv1d(in_channels=n_base * 4, out_channels=n_base, kernel_size=kernel_size, stride=stride,
                      padding=padding),
            nn.ReLU(),
            nn.AvgPool1d(kernel_size=2, stride=2)
        )
        self.conv4 = nn.Sequential(
            nn.Conv1d(in_channels=n_base, out_channels=embedding_dim, kernel_size=kernel_size, stride=stride,
                      padding=padding),
            nn.ReLU(),
            nn.AvgPool1d(kernel_size=2, stride=2)
        )

    def forward(self, x):
        x = self.dropout(x)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        return x
    
    
class LSTMEncoder(nn.Module):
    def __init__(
            self,
            embedding_dim: int = 512,
            hidden_size: int = 1024,
            n_layers: int = 4,
            dropout: float = 0.1,
            random_init: bool = False
    ):
        super().__init__()
        self.random_init = random_init
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=n_layers,
            dropout=dropout,
            batch_first=True,
            bidirectional=False
        )

    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)
        return out, h_n, c_n


class StackedBiLSTMEncoder(nn.Module):
    def __init__(
            self,
            embedding_dim: int = 512,
            hidden_size: int = 1024,
            n_layers: int = 4,
            dropout: float = 0.1,
            random_init: bool = False
    ):
        super().__init__()
        self.random_init = random_init
        # Init state
        if random_init:
            (h_0, c_0) = self.__init_state(n_layers=n_layers, hidden_size=hidden_size)
            self.init_state = (h_0, c_0)

        self.bilstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            bidirectional=True,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout
        )

    def forward(self, x):
        if self.random_init:
            out, (h_n, c_n) = self.bilstm(x, (self.init_state[0].detach(), self.init_state[1].detach()))
            return out, h_n, c_n
        else:
            out, (h_n, c_n) = self.bilstm(x)
            return out, h_n, c_n

    @staticmethod
    def __init_state(n_layers: int = 4, hidden_size: int = 1024):
        h_0 = torch.zeros(n_layers * 2, BATCH_SIZE, hidden_size).requires_grad_(True)
        c_0 = torch.zeros(n_layers * 2, BATCH_SIZE, hidden_size).requires_grad_(True)
        return h_0, c_0


class ParallelBiLSTMEncoder(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        pass


class Classifier(nn.Module):
    def __init__(self, num_class: int, d_model: int = 512, d_ff: int = 2048):
        super().__init__()
        self.ff1 = nn.Linear(in_features=d_model, out_features=d_ff)
        self.ff2 = nn.Linear(in_features=d_ff, out_features=num_class)
        self.act1 = nn.ReLU()
        # self.act2 = nn.Softmax()

    def forward(self, x):
        x = self.act1(self.ff1(x))
        # x = self.act2(self.ff2(x))
        x = self.ff2(x)
        return x


"""Transformer Model"""
class TransformerClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.input_embedding = InputEmbedding(
            vocab_size=config['vocab_size'],
            d_model=config['d_model']
        )
        self.positional_encoding = PositionalEncoding(
            d_model=config['d_model'],
            dropout=config['dropout'],
            max_len=config['max_len']
        )
        self.encoder = TransformerEncoder(
            d_model=config['d_model'],
            nhead=config['nhead'],
            num_layers=config['num_layers']
        )
        self.classifier = Classifier(
            d_model=config['d_model'],
            num_class=len(SP_LABELS)
        )

    def forward(self, x):
        x = self.input_embedding(x)
        x = self.positional_encoding(x)
        x = self.encoder(x)
        x = self.classifier(x)
        return x
    
"""Convolutional Model"""

class ConvolutionalClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.input_embedding = InputEmbedding(
            vocab_size=config['vocab_size'],
            d_model=config['d_model']
        )
        self.conv_encoder = ConvolutionalEncoder(
            embedding_dim=config['d_model'],
            kernel_size=config['kernel_size']
        )
        self.flatten = nn.Flatten()
        self.classifier = Classifier(num_class=len(SP_LABELS), d_model=1536)
        # self.dropout = nn.Dropout(p=0.1)

    def forward(self, x):
        x = self.input_embedding(x)
        x = torch.transpose(x, 1, 2)
        x = self.conv_encoder(x)
        x = self.flatten(x)
#         print(x.shape)
        x = self.classifier(x)
        return x

"""Stacked BiLSTM"""
class StackedBiLSTMClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.input_embedding = InputEmbedding(
            vocab_size=config['vocab_size'],
            d_model=config['d_model']
        )
        self.stacked_encoder = StackedBiLSTMEncoder(
            embedding_dim=config['d_model'],
            hidden_size=config['hidden_size'],
            n_layers=config['n_layers'],
            dropout=config['dropout'],

        )
        self.classifier = Classifier(num_class=len(SP_LABELS), d_model=config['hidden_size'] * 2)

    def forward(self, x):
        x = self.input_embedding(x)
        x, h_n, c_n = self.stacked_encoder(x)
        x = self.classifier(x[:, -1, :])
        return x

"""ProtBert Model"""
class ProtBertClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.bert = BertModel(config=config)
        # self.bert_encoder = get_peft_model(encoder, peft_config)
        self.classifier = Classifier(num_class=len(SP_LABELS), d_model=config.hidden_size)

    def forward(self, x):
        x = self.bert(x)
        x = x.last_hidden_state[:, 0, :]
        x = self.classifier(x)
        return x
    

"""LSTM Model"""
class LSTMClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.input_embedding = InputEmbedding(
            vocab_size=config['vocab_size'],
            d_model=config['d_model']
        )
        self.encoder = LSTMEncoder(
            embedding_dim=config['d_model'],
            hidden_size=config['hidden_size'],
            n_layers=config['n_layers'],
            dropout=config['dropout'],

        )
        self.classifier = Classifier(num_class=len(SP_LABELS), d_model=config['hidden_size'])

    def forward(self, x):
        x = self.input_embedding(x)
        x, h_n, c_n = self.encoder(x)
        x = self.classifier(x[:, -1, :])
        return x
    

"""CNN + Transformer Model"""
class CNNTransformerClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.input_embedding = InputEmbedding(
            vocab_size=config['vocab_size'],
            d_model=config['d_model']
        )
        self.conv_encoder = ConvolutionalEncoder(
            embedding_dim=config['d_model'],
            kernel_size=config['kernel_size']
        )
        self.positional_encoding = PositionalEncoding(
            d_model=config['d_model'],
            dropout=config['dropout'],
            max_len=config['max_len']
        )
        self.trans_encoder = TransformerEncoder(
            d_model=config['d_model'],
            nhead=config['nhead'],
            num_layers=config['num_layers']
        )
        self.classifier = Classifier(
            d_model=config['d_model'],
            num_class=len(SP_LABELS)
        )

    def forward(self, x):
        x = self.input_embedding(x)
        x = torch.transpose(x, 1, 2)
        x = self.conv_encoder(x)
        x = torch.transpose(x, 1, 2)
        x = self.positional_encoding(x)
        x = self.trans_encoder(x)
        x = self.classifier(x)
        return x


print("Finished")

Finished


In [9]:
# code Lightning Data Module
from pathlib import Path
from typing import Optional, Literal

import lightning as L
from lightning.pytorch.utilities.types import EVAL_DATALOADERS, TRAIN_DATALOADERS
from torch.utils.data import DataLoader

# from data.sp_dataset import SPDataset


class SPDataModule(L.LightningDataModule):
    def __init__(self, dataset_type: Literal["aa", "smiles"] = "smiles"):
        super().__init__()
        self.sp_dataset = None
        self.test_set = None
        self.val_set = None
        self.train_set = None
        self.dataset_type = dataset_type

    # def prepare_data(self) -> None:
    #     data_utils = DataUtils()
    #     data_utils.extract_raw_dataset_by_partition(raw_path=str(Path(ROOT_DIR) / configs.TRAIN_PATH))
    #     # data_utils.extract_raw_dataset_by_partition(raw_path=str(Path(ROOT_DIR) / configs.BENCHMARK_PATH),
    #     #                                             benchmark=True)

    def setup(self, stage: Optional[str] = None) -> None:
        if stage == "fit" or stage is None:
            # print(f"\nSetting up: Using {self.dataset_type}\n")
            train_path = [str(Path(ROOT_DIR) / 'data/sp_data/train_set_partition_0.json'),
                          str(Path(ROOT_DIR) / 'data/sp_data/train_set_partition_1.json')]
            val_path = [str(Path(ROOT_DIR) / 'data/sp_data/test_set_partition_0.json'),
                        str(Path(ROOT_DIR) / 'data/sp_data/test_set_partition_1.json')]
            self.train_set = SPDataset(json_paths=train_path, data_type=self.dataset_type)
            self.val_set = SPDataset(json_paths=val_path, data_type=self.dataset_type)
        elif stage == "test":
            test_path = [str(Path(ROOT_DIR) / 'data/sp_data/train_set_partition_2.json'),
                         str(Path(ROOT_DIR) / 'data/sp_data/test_set_partition_2.json')]
            self.test_set = SPDataset(json_paths=test_path, data_type=self.dataset_type)

    def train_dataloader(self) -> TRAIN_DATALOADERS:
        return DataLoader(self.train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=1,
                          persistent_workers=True)

    def val_dataloader(self) -> EVAL_DATALOADERS:
        return DataLoader(self.val_set, batch_size=BATCH_SIZE, shuffle=False)

    def test_dataloader(self) -> EVAL_DATALOADERS:
        return DataLoader(self.test_set, batch_size=BATCH_SIZE, shuffle=False)
    
print("Finished")

Finished


In [10]:
# code Lightning Module
import json
import os.path
from pathlib import Path
from typing import Literal

import lightning as L
import numpy as np
import pandas as pd
import torch
from torch import optim
from torch.nn import CrossEntropyLoss, Softmax
from torchmetrics import F1Score, MatthewsCorrCoef, AveragePrecision
from transformers import BertConfig, BertModel
from sklearn.metrics import classification_report

# import params
# from model.sp_bilstm import StackedBiLSTMClassifier
# from model.sp_cnn import ConvolutionalClassifier
# from model.sp_transformers import TransformerClassifier


# from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, matthews_corrcoef
# from torcheval.metrics.functional import multiclass_auroc, multiclass_auprc, multiclass_f1_score


class SPModule(L.LightningModule):
    def __init__(
            self,
            model_type: str = 'transformer',
            config_path: str | None = None

    ):
        super().__init__()
        # Effecient weights 
#         loss_weight = torch.tensor([0.5, 1.0, 1.0, 1.0, 1.0, 1.0], dtype=torch.float)
#         self.loss_fn = CrossEntropyLoss(weight=loss_weight)
        self.loss_fn = CrossEntropyLoss()
        self.model_type = model_type
        self.save_hyperparameters()

        # Load config (Remove if unnecessary)
        self.config = self.__load_model_config(model_type=model_type, config_path=config_path)

        # Load model
        self.model = self.__load_model(model_type=model_type, config_path=config_path)

        # Load metrics
        self.f1 = F1Score(task='multiclass', num_classes=len(SP_LABELS), average='micro')
        self.mcc = MatthewsCorrCoef(task='multiclass', num_classes=len(SP_LABELS))
        self.average_precision = AveragePrecision(task='multiclass', num_classes=len(SP_LABELS))

        # Outputs from training process
        self.validation_step_outputs_lb = []
        self.validation_step_outputs_pred = []
        self.best_val_loss = 1e6

        self.test_outputs_lb = []
        self.test_outputs_pred = []
        self.test_step_outputs = {}

    def forward(self, x):
        return self.model(x)

    def configure_optimizers(self):
        optimizer = optim.Adam(self.model.parameters(), lr=1e-4, weight_decay=0.1)
        return optimizer

    def base_step(self, batch, batch_idx):
        x, lb, kingdom = batch
        pred = self.model(x)
        loss = self.loss_fn(pred.float(), lb.float())
        return x, lb, pred, loss, kingdom

    def training_step(self, batch, batch_idx):
        _, _, pred, loss, _ = self.base_step(batch, batch_idx)
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        return loss

    def validation_step(self, batch, batch_idx):
        _, lb, pred, loss, _ = self.base_step(batch, batch_idx)
        self.validation_step_outputs_pred.append(pred)
        self.validation_step_outputs_lb.append(lb)
        self.log("val_loss", loss, prog_bar=True, sync_dist=True)
        # return loss

    def on_validation_epoch_end(self):
        all_pred = torch.concat(self.validation_step_outputs_pred, dim=0)
        all_lb = torch.concat(self.validation_step_outputs_lb, dim=0)

        val_loss = self.loss_fn(all_pred.float(), all_lb.float())
        if val_loss < self.best_val_loss:
            self.best_val_loss = val_loss
            # self.save_to_txt(all_pred)

        all_lb = torch.argmax(all_lb, dim=1)

        self.f1.update(all_pred, all_lb)
        self.mcc.update(all_pred, all_lb)
        self.average_precision.update(all_pred, all_lb)

        print(
            f"\nError on validation set: "
            f"best_val_loss: {self.best_val_loss}, "
            f"f1: {self.f1.compute()}, "
            f"mcc: {self.mcc.compute()}, "
            f"average_precision: {self.average_precision.compute()} \n"
        )

        self.validation_step_outputs_lb.clear()
        self.validation_step_outputs_pred.clear()
        self.f1.reset()
        self.mcc.reset()
        self.average_precision.reset()

    def test_step(self, batch, batch_idx):
        _, lb, pred, loss, kingdom = self.base_step(batch, batch_idx)

        # Update outputs for calculate metrics on each class
        _pred = torch.argmax(pred, dim=1)
        _lb = torch.argmax(lb, dim=1)
        self.test_outputs_pred.extend(_pred.detach().cpu().numpy())
        self.test_outputs_lb.extend(_lb.detach().cpu().numpy())

        # Update outputs for calculate metrics on each kingdom
        for i, k in enumerate(kingdom):
            if k not in self.test_step_outputs.keys():
                self.test_step_outputs[k] = {}
                self.test_step_outputs[k]["pred"] = []
                self.test_step_outputs[k]["lb"] = []

            outputs = self.test_step_outputs[k]
            self.test_step_outputs[k]['pred'] = torch.stack([*outputs["pred"], pred[i]])
            self.test_step_outputs[k]['lb'] = torch.stack([*outputs["lb"], lb[i]])
            # outputs["pred"].append(pred[i])
            # outputs["lb"].append(lb[i])

        # self.test_step_outputs_lb.append(lb)
        # self.test_step_outputs_pred.append(pred)
        # lb = torch.argmax(lb, dim=1)

    def on_test_end(self) -> None:
        # all_pred = torch.concat(self.test_step_outputs_lb, dim=0)
        # all_lb = torch.concat(self.test_step_outputs_lb, dim=0)
        # all_lb = torch.argmax(all_lb, dim=1)
        # all_test_outputs = {}
        
        # Apply argmax on these outputs (only for label) and evaluate the metric results
        print(classification_report(self.test_outputs_lb, self.test_outputs_pred))
        
        f1_test = []
        mcc_test = []
        average_precision_test = []
        for key, _ in KINGDOM.items():
            all_pred = self.test_step_outputs[key]['pred']
            all_lb = torch.argmax(self.test_step_outputs[key]['lb'], dim=1)

            # Print the statistic (the following function has ERROR about syntax)
            self.__save_results_to_txt(all_pred.detach().cpu(), all_lb.detach().cpu(), kingdom=key)

            # all_test_outputs[key] = {
            #     "pred": all_pred,
            #     "lb": all_lb
            # }

            self.f1.update(all_pred, all_lb)
            self.mcc.update(all_pred, all_lb)
            self.average_precision.update(all_pred, all_lb)
            f1_test.append(self.f1.compute().item())
            mcc_test.append(self.mcc.compute().item())
            average_precision_test.append(self.average_precision.compute().item())

            print(
                f'\nError on test set of {key}: '
                f'f1: {self.f1.compute()}, '
                f'mcc: {self.mcc.compute()}, '
                f'average_precision: {self.average_precision.compute()} \n'
            )

            self.f1.reset()
            self.mcc.reset()
            self.average_precision.reset()

        metric_dict = {
            "kingdom": KINGDOM.keys(),
            "F1 Score": f1_test,
            "MCC Score": mcc_test,
            "Average Precision Score": average_precision_test
        }

        self.__save_metrics_to_csv(metric_dict)

    @staticmethod
    def __save_results_to_txt(test_prediction_results, test_true_results, kingdom):
        if not os.path.exists(str(Path(KAGGLE_DIR) / "out")):
            os.makedirs('out', exist_ok=True)
        softmax = Softmax()
        pred_path = f'out/{kingdom}_test_prediction_results.txt'
        true_path = f'out/{kingdom}_true_prediction_results.txt'

        # print(test_prediction_results)

        np.savetxt(str(Path(KAGGLE_DIR) / pred_path), softmax(test_prediction_results), fmt="%.4f")
        np.savetxt(str(Path(KAGGLE_DIR) / true_path), test_true_results, fmt="%d")

    def __save_metrics_to_csv(self, metric_dict):
        version = 0
        while os.path.exists(str(Path(KAGGLE_DIR) / f'out/{self.model_type}_metrics_results_{version}.csv')):
            version += 1

        df = pd.DataFrame().from_dict(metric_dict)
        df.to_csv(str(Path(KAGGLE_DIR) / f'out/{self.model_type}_metrics_results_{version}.csv'), index_label="No")

    @staticmethod
    def __save_config():
        pass

    @staticmethod
    def __load_model_config(model_type, config_path: str | None = None):
        if model_type == "bert_pretrained":
            config = BertConfig().from_pretrained("Rostlab/prot_bert")
            return config
        elif model_type == "bert":
            with open(str(Path(ROOT_DIR) / f'configs/{model_type}_config_default.json')) as f:
                data = json.load(f)
                config = BertConfig(**data)
                return config
        else:
            if config_path is None:
                with open(str(Path(ROOT_DIR) / f'configs/{model_type}_config_default.json')) as f:
                    config = json.load(f)
                    return config
            else:
                if not os.path.exists(str(Path(ROOT_DIR) / config_path)):
                    raise FileNotFoundError("Config file does not exist")
                else:
                    with open(config_path, 'r') as f:
                        config = json.load(f)
                        return config

    def __load_model(self, model_type, config_path: str | None = None):
        config = self.__load_model_config(model_type, config_path)
        if model_type == 'transformer':
            return TransformerClassifier(config)
        elif model_type == 'cnn':
            return ConvolutionalClassifier(config)
        elif model_type == 'st_bilstm':
            return StackedBiLSTMClassifier(config)
        elif model_type == "bert_pretrained" or model_type == "bert":
            return BertModel(config)
        elif model_type == "lstm":
            return LSTMClassifier(config)
        elif model_type == "cnn_trans":
            return CNNTransformerClassifier(config)
        else:
            return ValueError("Unknown model type")

    @staticmethod
    def __unfreeze(layer):
        pass

    @staticmethod
    def __freeze(layer):
        pass
    

print("Finished")


Finished


In [15]:
# Training and Validation
from lightning.pytorch.loggers.tensorboard import TensorBoardLogger

logger = TensorBoardLogger(save_dir=str(Path(KAGGLE_DIR) / DEFAULT_LOG_DIR), name='tensorboard_logs')

sp_module = SPModule(model_type=MODEL)
sp_data_module = SPDataModule(dataset_type=DATA)
trainer = L.Trainer(
    devices='auto',
    accelerator='auto',
    max_epochs=EPOCHS,
    val_check_interval=1.0,
    logger=logger,
    callbacks=[model_checkpoint, early_stopping]
)

trainer.fit(sp_module, datamodule=sp_data_module)

INFO: GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO: IPU available: False, using: 0 IPUs
INFO: HPU available: False, using: 0 HPUs
INFO: `Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..
/opt/conda/lib/python3.10/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:653: Checkpoint directory /kaggle/working exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name              | Type                       | Params
-----------------------------------------------------------------
0 | loss_fn           | CrossEntropyLoss           | 0     
1 | model             | LSTMClassifier             | 33.6 M
2 | f1                | MulticlassF1Score          | 0     
3 | mcc               | MulticlassMatthewsCorrCoef | 0     
4 | average_precision | MulticlassAveragePrecision | 0     
-----------------------------------------------------------------
33.6 M   

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/opt/conda/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:441: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
/tmp/ipykernel_34/16684519.py:38: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(input_ids, dtype=torch.int64), torch.tensor(label, dtype=torch.int64), kingdom
/opt/conda/lib/python3.10/site-packages/lightning/pytorch/utilities/data.py:77: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 8. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.



Error on validation set: best_val_loss: 1.7862277030944824, f1: 0.0, mcc: 0, average_precision: 0.37150681018829346 



/opt/conda/lib/python3.10/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Average precision score for one or more classes was `nan`. Ignoring these classes in macro-average
  warnings.warn(*args, **kwargs)  # noqa: B028
/opt/conda/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:441: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


...

Validation: |          | 0/? [00:00<?, ?it/s]


Error on validation set: best_val_loss: 0.8223016262054443, f1: 0.7563909888267517, mcc: 0, average_precision: 0.18890058994293213 



Validation: |          | 0/? [00:00<?, ?it/s]


Error on validation set: best_val_loss: 0.8223016262054443, f1: 0.7563909888267517, mcc: 0, average_precision: 0.19440576434135437 



/opt/conda/lib/python3.10/site-packages/lightning/pytorch/trainer/call.py:54: Detected KeyboardInterrupt, attempting graceful shutdown...


In [ ]:
# # Testing
# sp_data_module = SPDataModule(dataset_type=DATA)
# sp_module = SPModule.load_from_checkpoint(str(Path(BERT_CKPT) / f'bert_pretrained_epoch=1_kaggle.ckpt'))
# trainer = L.Trainer(
#     devices='auto',
#     accelerator='auto',
#     max_epochs=EPOCHS,
#     val_check_interval=1.0,
#     logger=False
# )

# trainer.test(sp_module, datamodule=sp_data_module)

In [ ]:
# %reload_ext tensorboard
# %tensorboard --logdir logs --bind_all